# Visualization

Exports dataframes and maps that are not used for analysis, and are directly used to generate figures.


* **Figures:**
    * `Figure 2`: Asymptotes per aggregation level
    * `Figure 4`: Map of expected carbon accumulation by 2050
    * `Extended Data Fig. 1`: Biomass with EU TMF-obtained secondary forest age data
    * `Extended Data Fig. 2`: Mature Forest Biomass - distance to edge
    * `Extended Data Fig. 3`: Surrounding mature forest biomass across the Amazon
    * `Extended Data Fig. 4`: Why it is important to remove secondary forest edges

In [1]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

mapbiomas, lulc = desired_lulc()

import nbimporter # lets you import notebooks like regular modules
%run objects.ipynb

In [2]:
def export_image_toDrive(image, description):
    task = ee.batch.Export.image.toDrive(
        image = image,
        description = description,
        fileNamePrefix = description,
        region = roi,
        scale = image.projection().nominalScale(),
        maxPixels = 1e13,
        crs = 'EPSG:4326',
        fileFormat = 'GeoTIFF'
    )
    task.start()

## Figure 2: Asymptotes per aggregation level

In [3]:
nearest_mature = ee.Image(f"{data_folder}/nearest_mature").selfMask()

quarters_ecoreg_biomass = ee.Image(f"{data_folder}/quarters_ecoreg_biomass").select("quarter_biomass").selfMask()

biomes = ee.Image(f"{data_folder}/categorical").select("biome")
biomes_mask = biomes.eq(1).rename("biome_mask").selfMask()

# export_image(nearest_mature, "nearest_mature")
# export_image(quarters_ecoreg_biomass, "quarters_ecoreg_biomass")
# export_image(biomes_mask, "biome_mask")

## Figure 3: Lag per ecoregion

In [4]:
ecoreg = ee.FeatureCollection('RESOLVE/ECOREGIONS/2017')

vals = ee.FeatureCollection([
    ee.Feature(None, {'ecoreg': 476, 'lag': 41.7705136155675}),
    ee.Feature(None, {'ecoreg': 481, 'lag': 29.5895220291191}),
    ee.Feature(None, {'ecoreg': 507, 'lag': 27.6198740056028}),
    ee.Feature(None, {'ecoreg': 508, 'lag': 22.5976655714891}),
    ee.Feature(None, {'ecoreg': 518, 'lag': 29.4543147641033}),
])

join = ee.Join.inner()
filter_eq = ee.Filter.equals(leftField='ECO_ID', rightField='ecoreg')

# generates a join object - a feature collection without geometries
joined = join.apply(ecoreg, vals, filter_eq)

def merge_feature(f):
    left = ee.Feature(f.get('primary'))
    right = ee.Feature(f.get('secondary'))
    return left.set('lag', right.get('lag')).intersection(amazon_geom, ee.ErrorMargin(1))

ecoreg_lag = ee.FeatureCollection(joined.map(merge_feature)).select(["lag", 'ECO_ID'])

task = ee.batch.Export.table.toDrive(
    collection = ecoreg_lag,
    description = "ecoreg_lag",
    fileFormat = 'SHP'
)
# task.start()

In [6]:
age = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1")
    .select("secondary_vegetation_age_2020")
    .rename("age"))

grid = ee.FeatureCollection("projects/forestregrowth/assets/grid_10k_amazon_secondary_edge_removed")

map = geemap.Map()
map.addLayer(ecoreg)
map.addLayer(age)
map.addLayer(grid)
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

## Figure 4: Map of expected carbon accumulation by 2050

The resulting raster is then prepared for display and export as image on QGIS.

In [ ]:
pred_lag_2050_fc = ee.FeatureCollection("projects/amazon-forest-regrowth/assets/pred_2050_secondary")

pred_lag_2050 = pred_lag_2050_fc.reduceToImage(
            properties = ["pred"],
            reducer = ee.Reducer.first()
        ).reproject(
            crs = 'EPSG:4326',  # or match your source CRS
            scale = 100
        ).rename("pred")

# Aggregate the high-resolution pixels into the 10 km grid
pred_lag_2050 = pred_lag_2050.reduceResolution(
    reducer = ee.Reducer.median(),
    maxPixels = 65535,
    bestEffort = True
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("pred_lag_2050_10k")

export_image_toDrive(pred_lag_2050, "pred_lag_2050_secondary")

## Extended Data Fig. 1: Biomass with EU TMF-obtained secondary forest age data

In [ ]:
# Load the image collections
transition = ee.ImageCollection('projects/JRC/TMF/v1_2023/TransitionMap_Subtypes').mosaic().clip(roi)
annual_changes = ee.ImageCollection('projects/JRC/TMF/v1_2023/AnnualChanges').mosaic().clip(roi)

# Define regrowth and degraded conditions
regrowth = transition.gte(31).And(transition.lte(33))

# Initialize AgeRegrowth and AgeDegraded
tmf = ee.Image.constant(0)

# Calculate AgeRegrowth
for i in range(1990, last_year):
    year = 'Dec' + str(i)
    annual_changes_year = annual_changes.select(year)
    condition = annual_changes_year.eq(4).And(regrowth) # were regrowing then AND are regrowing now
    tmf = tmf.add(condition.eq(1))

tmf = tmf.selfMask().rename(f"tmf_{last_year}")


ESA_CCI = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').select("AGB").mean().rename("biomass")

ESA_CCI_resampled = ESA_CCI.reduceResolution(
        reducer= ee.Reducer.mean(),
    ).reproject(
        crs=tmf.projection(),
        scale=tmf.projection().nominalScale()
    )

tmf_mask = tmf.gt(0).selfMask().rename("tmf_mask")

tmf_ESA = ESA_CCI.addBands(tmf).addBands(tmf_mask)


tmf_ESA_fc = tmf_ESA.stratifiedSample(numPoints = 100,
                                      classBand = "tmf",
                                      dropNulls = True)

task = ee.batch.Export.table.toDrive(collection = tmf_ESA_fc, fileFormat="CSV")
# task.start()


## Extended Data Fig. 2: Mature Forest Biomass - distance to edge

Shows the biomass is systematically lower closer to the edge due to edge effects/disturbance.

In [22]:
biomes = ee.Image(f"{data_folder}/categorical").select("biome")
biomes_mask = biomes.eq(1).rename("biome_mask")

lulc = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))

mature_mask = lulc.eq(3).reduce(ee.Reducer.allNonZero()).selfMask().updateMask(biomes_mask).rename("mask")

distance_to_mature_edge = ee.Image(f"{data_folder}/distance_to_mature_edge").updateMask(mature_mask).rename("dist")

biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').mean().select("AGB").rename("biomass")

mature_biomass_distance = biomass.addBands(distance_to_mature_edge, mature_mask)

# properties_to_export = mature_biomass_distance.bandNames().getInfo()

# sampled_pixels = mature_biomass_distance.stratifiedSample(
#     numPoints = 10000,
#     classBand = 'mask',
#     region = biomes_mask.geometry(),
#     dropNulls = True
# )

# # Export task to Google Drive
# task = ee.batch.Export.table.toDrive(
#     collection = sampled_pixels,
#     description = 'mature_biomass_distance',
#     fileFormat = "CSV",
#     selectors = [p for p in properties_to_export if p not in ['system:index', '.geo']]
# )
# task.start()

# map = geemap.Map()
# map.addLayer(mature_biomass_distance, {}, 'mature_biomass_distance')
# map

## Extended Data Figure 3: Surrounding mature forest biomass across the Amazon

In [ ]:
mature_biomass = ee.Image(f"{data_folder}/mature_biomass")

# Aggregate the high-resolution pixels into the 10 km grid
mature_biomass = mature_biomass.reduceResolution(
    reducer = ee.Reducer.median(),
    maxPixels = 65535,
    bestEffort = True
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("mature_biomass")

export_image_toDrive(mature_biomass, "mature_biomass")

## Extended Data Figure 4: Why it is important to remove secondary forest edges